# CUDA Matmul Kernel Test — Naive vs Tiled
Open this notebook in Google Colab (Runtime > Change runtime type > GPU)

In [ ]:
!nvidia-smi

In [ ]:
%%writefile matmul.cu
#include <cuda_runtime.h>
#include <stdio.h>

__global__ void matmul_kernel(float *A, float *B, float *C, int M, int K, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < M && col < N) {
        float sum = 0.0f;
        for (int k = 0; k < K; k++) {
            sum += A[row * K + k] * B[k * N + col];
        }
        C[row * N + col] = sum;
    }
}

extern "C" {

void cuda_matmul(float *host_A, float *host_B, float *host_C, int M, int K, int N) {
    float *d_A, *d_B, *d_C;

    size_t size_A = M * K * sizeof(float);
    size_t size_B = K * N * sizeof(float);
    size_t size_C = M * N * sizeof(float);

    cudaMalloc(&d_A, size_A);
    cudaMalloc(&d_B, size_B);
    cudaMalloc(&d_C, size_C);

    cudaMemcpy(d_A, host_A, size_A, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, host_B, size_B, cudaMemcpyHostToDevice);

    dim3 threads(16, 16);
    dim3 blocks((N + 15) / 16, (M + 15) / 16);

    matmul_kernel<<<blocks, threads>>>(d_A, d_B, d_C, M, K, N);

    cudaMemcpy(host_C, d_C, size_C, cudaMemcpyDeviceToHost);

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
}

}

In [ ]:
!nvcc -shared -Xcompiler -fPIC -o matmul.so matmul.cu
print('Compiled successfully!')

In [ ]:
import ctypes
import numpy as np

lib = ctypes.CDLL('./matmul.so')
lib.cuda_matmul.argtypes = [
    ctypes.POINTER(ctypes.c_float),
    ctypes.POINTER(ctypes.c_float),
    ctypes.POINTER(ctypes.c_float),
    ctypes.c_int,
    ctypes.c_int,
    ctypes.c_int,
]
lib.cuda_matmul.restype = None

def cuda_matmul(a, b):
    a = np.ascontiguousarray(a, dtype=np.float32)
    b = np.ascontiguousarray(b, dtype=np.float32)
    M, K = a.shape
    N = b.shape[1]
    c = np.empty((M, N), dtype=np.float32)
    lib.cuda_matmul(
        a.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),
        b.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),
        c.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),
        M, K, N,
    )
    return c

print('Library loaded!')

In [ ]:
print('=== Test 1: Small matrix (3x4) @ (4x5) ===')
A = np.random.randn(3, 4).astype(np.float32)
B = np.random.randn(4, 5).astype(np.float32)

gpu_result = cuda_matmul(A, B)
cpu_result = A @ B

max_diff = np.max(np.abs(gpu_result - cpu_result))
print(f'GPU result:\n{gpu_result}')
print(f'CPU result:\n{cpu_result}')
print(f'Max difference: {max_diff:.10f}')
print(f'PASS: {max_diff < 1e-5}')

print()
print('=== Test 2: Bigger matrix (128x256) @ (256x64) ===')
A2 = np.random.randn(128, 256).astype(np.float32)
B2 = np.random.randn(256, 64).astype(np.float32)

gpu_result2 = cuda_matmul(A2, B2)
cpu_result2 = A2 @ B2

max_diff2 = np.max(np.abs(gpu_result2 - cpu_result2))
print(f'Max difference: {max_diff2:.10f}')
print(f'PASS: {max_diff2 < 1e-4}')

print()
print('=== Test 3: Non-square (1x1024) @ (1024x1) ===')
A3 = np.random.randn(1, 1024).astype(np.float32)
B3 = np.random.randn(1024, 1).astype(np.float32)

gpu_result3 = cuda_matmul(A3, B3)
cpu_result3 = A3 @ B3

max_diff3 = np.max(np.abs(gpu_result3 - cpu_result3))
print(f'GPU: {gpu_result3.flatten()[0]:.6f}, CPU: {cpu_result3.flatten()[0]:.6f}')
print(f'Max difference: {max_diff3:.10f}')
print(f'PASS: {max_diff3 < 1e-4}')

In [ ]:
import time

sizes = [64, 256, 512, 1024, 2048]

print(f'{"Size":>8} | {"NumPy (CPU)":>12} | {"CUDA (GPU)":>12} | {"Speedup":>8}')
print('-' * 50)

for n in sizes:
    A = np.random.randn(n, n).astype(np.float32)
    B = np.random.randn(n, n).astype(np.float32)

    cuda_matmul(A, B)

    start = time.time()
    for _ in range(10):
        cpu_result = A @ B
    cpu_time = (time.time() - start) / 10

    start = time.time()
    for _ in range(10):
        gpu_result = cuda_matmul(A, B)
    gpu_time = (time.time() - start) / 10

    speedup = cpu_time / gpu_time if gpu_time > 0 else 0
    print(f'{n:>8} | {cpu_time*1000:>9.2f} ms | {gpu_time*1000:>9.2f} ms | {speedup:>7.1f}x')

## Tiled Matmul — Shared Memory Optimization

In [ ]:
%%writefile matmul_tiled.cu
#include <cuda_runtime.h>
#include <stdio.h>

#define TILE 16

__global__ void matmul_tiled_kernel(float *A, float *B, float *C, int M, int K, int N) {
    __shared__ float tile_A[TILE][TILE];
    __shared__ float tile_B[TILE][TILE];

    int row = blockIdx.y * TILE + threadIdx.y;
    int col = blockIdx.x * TILE + threadIdx.x;

    float sum = 0.0f;

    int num_tiles = (K + TILE - 1) / TILE;

    for (int t = 0; t < num_tiles; t++) {
        int a_col = t * TILE + threadIdx.x;
        if (row < M && a_col < K)
            tile_A[threadIdx.y][threadIdx.x] = A[row * K + a_col];
        else
            tile_A[threadIdx.y][threadIdx.x] = 0.0f;

        int b_row = t * TILE + threadIdx.y;
        if (b_row < K && col < N)
            tile_B[threadIdx.y][threadIdx.x] = B[b_row * N + col];
        else
            tile_B[threadIdx.y][threadIdx.x] = 0.0f;

        __syncthreads();

        for (int k = 0; k < TILE; k++) {
            sum += tile_A[threadIdx.y][k] * tile_B[k][threadIdx.x];
        }

        __syncthreads();
    }

    if (row < M && col < N) {
        C[row * N + col] = sum;
    }
}

extern "C" {

void cuda_matmul_tiled(float *host_A, float *host_B, float *host_C, int M, int K, int N) {
    float *d_A, *d_B, *d_C;

    size_t size_A = M * K * sizeof(float);
    size_t size_B = K * N * sizeof(float);
    size_t size_C = M * N * sizeof(float);

    cudaMalloc(&d_A, size_A);
    cudaMalloc(&d_B, size_B);
    cudaMalloc(&d_C, size_C);

    cudaMemcpy(d_A, host_A, size_A, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, host_B, size_B, cudaMemcpyHostToDevice);

    dim3 threads(TILE, TILE);
    dim3 blocks((N + TILE - 1) / TILE, (M + TILE - 1) / TILE);

    matmul_tiled_kernel<<<blocks, threads>>>(d_A, d_B, d_C, M, K, N);

    cudaMemcpy(host_C, d_C, size_C, cudaMemcpyDeviceToHost);

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
}

}

In [ ]:
!nvcc -shared -Xcompiler -fPIC -o matmul_tiled.so matmul_tiled.cu
print('Tiled kernel compiled!')

In [ ]:
lib_tiled = ctypes.CDLL('./matmul_tiled.so')
lib_tiled.cuda_matmul_tiled.argtypes = [
    ctypes.POINTER(ctypes.c_float),
    ctypes.POINTER(ctypes.c_float),
    ctypes.POINTER(ctypes.c_float),
    ctypes.c_int,
    ctypes.c_int,
    ctypes.c_int,
]
lib_tiled.cuda_matmul_tiled.restype = None

def cuda_matmul_tiled(a, b):
    a = np.ascontiguousarray(a, dtype=np.float32)
    b = np.ascontiguousarray(b, dtype=np.float32)
    M, K = a.shape
    N = b.shape[1]
    c = np.empty((M, N), dtype=np.float32)
    lib_tiled.cuda_matmul_tiled(
        a.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),
        b.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),
        c.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),
        M, K, N,
    )
    return c

print('Tiled library loaded!')

In [ ]:
print('=== Correctness check: tiled vs NumPy ===')
A_test = np.random.randn(256, 512).astype(np.float32)
B_test = np.random.randn(512, 128).astype(np.float32)

tiled_result = cuda_matmul_tiled(A_test, B_test)
cpu_result = A_test @ B_test

max_diff = np.max(np.abs(tiled_result - cpu_result))
print(f'Max difference: {max_diff:.10f}')
print(f'PASS: {max_diff < 1e-3}')

In [ ]:
sizes = [64, 256, 512, 1024, 2048, 4096]

print(f'{"Size":>8} | {"NumPy (CPU)":>12} | {"Naive GPU":>12} | {"Tiled GPU":>12} | {"Naive spd":>10} | {"Tiled spd":>10}')
print('-' * 80)

for n in sizes:
    A = np.random.randn(n, n).astype(np.float32)
    B = np.random.randn(n, n).astype(np.float32)

    cuda_matmul(A, B)
    cuda_matmul_tiled(A, B)

    reps = 20 if n <= 1024 else 5

    start = time.time()
    for _ in range(reps):
        _ = A @ B
    cpu_time = (time.time() - start) / reps

    start = time.time()
    for _ in range(reps):
        _ = cuda_matmul(A, B)
    naive_time = (time.time() - start) / reps

    start = time.time()
    for _ in range(reps):
        _ = cuda_matmul_tiled(A, B)
    tiled_time = (time.time() - start) / reps

    naive_spd = cpu_time / naive_time if naive_time > 0 else 0
    tiled_spd = cpu_time / tiled_time if tiled_time > 0 else 0

    print(f'{n:>8} | {cpu_time*1000:>9.2f} ms | {naive_time*1000:>9.2f} ms | {tiled_time*1000:>9.2f} ms | {naive_spd:>9.1f}x | {tiled_spd:>9.1f}x')

## GRU Kernel Test

In [ ]:
%%writefile gru.cu
#include <cuda_runtime.h>
#include <math.h>

__device__ float sigmoid(float x) {
    return 1.0f / (1.0f + expf(-x));
}

__global__ void gru_forward_kernel(
    float *x, float *h_prev,
    float *W_r, float *b_r,
    float *W_z, float *b_z,
    float *W_c, float *b_c,
    float *h_out,
    int batch, int input_size, int hidden_size
) {
    extern __shared__ float shared[];
    float *s_r = shared;

    int b_idx = blockIdx.x;
    int j = threadIdx.x;

    if (b_idx >= batch || j >= hidden_size) return;

    float r_val = b_r[j];
    for (int k = 0; k < input_size; k++) {
        r_val += x[b_idx * input_size + k] * W_r[k * hidden_size + j];
    }
    for (int k = 0; k < hidden_size; k++) {
        r_val += h_prev[b_idx * hidden_size + k] * W_r[(input_size + k) * hidden_size + j];
    }
    r_val = sigmoid(r_val);

    s_r[j] = r_val;
    __syncthreads();

    float z_val = b_z[j];
    for (int k = 0; k < input_size; k++) {
        z_val += x[b_idx * input_size + k] * W_z[k * hidden_size + j];
    }
    for (int k = 0; k < hidden_size; k++) {
        z_val += h_prev[b_idx * hidden_size + k] * W_z[(input_size + k) * hidden_size + j];
    }
    z_val = sigmoid(z_val);

    float c_val = b_c[j];
    for (int k = 0; k < input_size; k++) {
        c_val += x[b_idx * input_size + k] * W_c[k * hidden_size + j];
    }
    for (int k = 0; k < hidden_size; k++) {
        float rh = s_r[k] * h_prev[b_idx * hidden_size + k];
        c_val += rh * W_c[(input_size + k) * hidden_size + j];
    }
    c_val = tanhf(c_val);

    float h_j = h_prev[b_idx * hidden_size + j];
    h_out[b_idx * hidden_size + j] = z_val * h_j + (1.0f - z_val) * c_val;
}

extern "C" {

void cuda_gru_forward(
    float *x, float *h_prev,
    float *W_r, float *b_r,
    float *W_z, float *b_z,
    float *W_c, float *b_c,
    float *h_out,
    int batch, int input_size, int hidden_size
) {
    float *d_x, *d_h_prev, *d_h_out;
    float *d_W_r, *d_b_r, *d_W_z, *d_b_z, *d_W_c, *d_b_c;

    int combined_size = input_size + hidden_size;

    size_t sz_x = batch * input_size * sizeof(float);
    size_t sz_h = batch * hidden_size * sizeof(float);
    size_t sz_W = combined_size * hidden_size * sizeof(float);
    size_t sz_b = hidden_size * sizeof(float);

    cudaMalloc(&d_x, sz_x);
    cudaMalloc(&d_h_prev, sz_h);
    cudaMalloc(&d_h_out, sz_h);
    cudaMalloc(&d_W_r, sz_W);
    cudaMalloc(&d_b_r, sz_b);
    cudaMalloc(&d_W_z, sz_W);
    cudaMalloc(&d_b_z, sz_b);
    cudaMalloc(&d_W_c, sz_W);
    cudaMalloc(&d_b_c, sz_b);

    cudaMemcpy(d_x, x, sz_x, cudaMemcpyHostToDevice);
    cudaMemcpy(d_h_prev, h_prev, sz_h, cudaMemcpyHostToDevice);
    cudaMemcpy(d_W_r, W_r, sz_W, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b_r, b_r, sz_b, cudaMemcpyHostToDevice);
    cudaMemcpy(d_W_z, W_z, sz_W, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b_z, b_z, sz_b, cudaMemcpyHostToDevice);
    cudaMemcpy(d_W_c, W_c, sz_W, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b_c, b_c, sz_b, cudaMemcpyHostToDevice);

    dim3 blocks(batch);
    dim3 threads(hidden_size);
    size_t shared_mem = hidden_size * sizeof(float);

    gru_forward_kernel<<<blocks, threads, shared_mem>>>(
        d_x, d_h_prev,
        d_W_r, d_b_r,
        d_W_z, d_b_z,
        d_W_c, d_b_c,
        d_h_out,
        batch, input_size, hidden_size
    );

    cudaMemcpy(h_out, d_h_out, sz_h, cudaMemcpyDeviceToHost);

    cudaFree(d_x);
    cudaFree(d_h_prev);
    cudaFree(d_h_out);
    cudaFree(d_W_r);
    cudaFree(d_b_r);
    cudaFree(d_W_z);
    cudaFree(d_b_z);
    cudaFree(d_W_c);
    cudaFree(d_b_c);
}

}

In [ ]:
!nvcc -shared -Xcompiler -fPIC -o gru.so gru.cu
print('GRU kernel compiled!')

In [ ]:
gru_lib = ctypes.CDLL('./gru.so')
gru_lib.cuda_gru_forward.argtypes = [
    ctypes.POINTER(ctypes.c_float)] * 9 + [ctypes.c_int] * 3
gru_lib.cuda_gru_forward.restype = None

def ptr(arr):
    return arr.ctypes.data_as(ctypes.POINTER(ctypes.c_float))

def cuda_gru(x, h_prev, W_r, b_r, W_z, b_z, W_c, b_c):
    batch, input_size = x.shape
    hidden_size = h_prev.shape[1]
    h_out = np.empty((batch, hidden_size), dtype=np.float32)
    gru_lib.cuda_gru_forward(
        ptr(x), ptr(h_prev),
        ptr(W_r), ptr(b_r), ptr(W_z), ptr(b_z), ptr(W_c), ptr(b_c),
        ptr(h_out),
        batch, input_size, hidden_size,
    )
    return h_out

def numpy_gru(x, h_prev, W_r, b_r, W_z, b_z, W_c, b_c):
    combined = np.concatenate([x, h_prev], axis=1)
    r = 1.0 / (1.0 + np.exp(-(combined @ W_r + b_r)))
    z = 1.0 / (1.0 + np.exp(-(combined @ W_z + b_z)))
    combined_r = np.concatenate([x, r * h_prev], axis=1)
    candidate = np.tanh(combined_r @ W_c + b_c)
    h_new = z * h_prev + (1 - z) * candidate
    return h_new

print('GRU functions defined!')

In [ ]:
batch = 16
input_size = 64
hidden_size = 128

x = np.random.randn(batch, input_size).astype(np.float32)
h_prev = np.random.randn(batch, hidden_size).astype(np.float32)

combined_size = input_size + hidden_size
W_r = np.random.randn(combined_size, hidden_size).astype(np.float32) * 0.1
b_r = np.zeros(hidden_size, dtype=np.float32)
W_z = np.random.randn(combined_size, hidden_size).astype(np.float32) * 0.1
b_z = np.zeros(hidden_size, dtype=np.float32)
W_c = np.random.randn(combined_size, hidden_size).astype(np.float32) * 0.1
b_c = np.zeros(hidden_size, dtype=np.float32)

gpu_h = cuda_gru(x, h_prev, W_r, b_r, W_z, b_z, W_c, b_c)
cpu_h = numpy_gru(x, h_prev, W_r, b_r, W_z, b_z, W_c, b_c)

max_diff = np.max(np.abs(gpu_h - cpu_h))
print(f'=== GRU Correctness Test ===')
print(f'batch={batch}, input={input_size}, hidden={hidden_size}')
print(f'Max difference: {max_diff:.8f}')
print(f'PASS: {max_diff < 1e-4}')
print()

print(f'GPU h_out[:2, :5]:\n{gpu_h[:2, :5]}')
print(f'CPU h_out[:2, :5]:\n{cpu_h[:2, :5]}')
print()

sizes = [(8, 32, 64), (16, 64, 128), (32, 128, 256), (64, 256, 512)]
print(f'{"Config":>20} | {"NumPy":>10} | {"CUDA":>10} | {"Speedup":>8}')
print('-' * 55)

for b, inp, hid in sizes:
    x_t = np.random.randn(b, inp).astype(np.float32)
    h_t = np.random.randn(b, hid).astype(np.float32)
    cs = inp + hid
    Wr = np.random.randn(cs, hid).astype(np.float32) * 0.1
    br = np.zeros(hid, dtype=np.float32)
    Wz = np.random.randn(cs, hid).astype(np.float32) * 0.1
    bz = np.zeros(hid, dtype=np.float32)
    Wc = np.random.randn(cs, hid).astype(np.float32) * 0.1
    bc = np.zeros(hid, dtype=np.float32)

    cuda_gru(x_t, h_t, Wr, br, Wz, bz, Wc, bc)

    reps = 100
    start = time.time()
    for _ in range(reps):
        numpy_gru(x_t, h_t, Wr, br, Wz, bz, Wc, bc)
    cpu_time = (time.time() - start) / reps

    start = time.time()
    for _ in range(reps):
        cuda_gru(x_t, h_t, Wr, br, Wz, bz, Wc, bc)
    gpu_time = (time.time() - start) / reps

    spd = cpu_time / gpu_time if gpu_time > 0 else 0
    label = f'b={b} in={inp} h={hid}'
    print(f'{label:>20} | {cpu_time*1000:>7.3f} ms | {gpu_time*1000:>7.3f} ms | {spd:>7.1f}x')